# Part 2: Building out Makemore

In [1]:
words = open('names.txt', 'r').read().splitlines()

In [2]:
chars = sorted(list(set(''.join(words))))

stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {s:i for i, s in stoi.items()}

In [3]:
import torch

In [4]:
# First, we have to prep the data to be fed into the neural net.
# How do we do that? We take two tensors: input and target. So, input is the tensor that gets fed into the NN. And, target is the one that we compare the results against.
# How do we get that? We take each bi-gram: (ch1, ch2). ch1 is going to be part of the input tensor, and at the corresponding position in the target tensor, we have ch2.

xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [5]:
xs

tensor([ 0,  5, 13,  ..., 25, 26, 24])

In [6]:
import torch.nn.functional as F

In [7]:
# now, we can't just pass this in to the NN, because our weights are going to be a 27x27 matrix
# so, we one-hot encode the input tensor, so that in each input tensor, only the index corresponding to the character is 1, and the rest are 0.
# in the W matrix, each column is a single neurons inputs (i.e the weights coming into it, from all 27 input tensors).
# so, when matrix mult takes place, the input tensor of shape (1, 27) dot products against each column of W, i.e. against all of the weights of all inputs to that neuron, but the 0's in the one-hot encoding cancel them out, and only the weight corresponding to the 1 in the input tensor is taken.
# this way, all 27 columns are parsed, and we get an output of shape (1, 27) from a single input tensor.

x_oh = F.one_hot(xs, num_classes=27).float()

In [8]:
# Now, we want to initialize the W matrix

W = torch.randn((27, 27), requires_grad=True)

In [9]:
x_oh.shape

torch.Size([228146, 27])

In [10]:
# now, we want to wire up the input to the W matrix, and get the output.

logits = x_oh @ W  # matrix mult

# these two lines below are the softmax function.
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)

In [11]:
# now, we come to the loss function. How are we going to implement this?
# we have the softmax prob distributions for each input tensor, and a corresponding output tensor too (probs[i])
# the output tensor (probs[i]) will serve as the prediction, but what is the target. The target is ys, i.e. we know what the correct next character is.
# So, we know the index of the correct next char in the bi-gram at ys. So, we will check the prob assigned to it by the NN using the index. It's a simple lookup.
# The point is that we want to maximize the probability on this character. So, based on this we average out the NLL across all of the input-output row pairs, and then, perform backprop.

nlls = torch.zeros(5)

for i in range(5):
    x = xs[i].item() # input char index
    y = ys[i].item() # label char index
    x_char = itos[x]
    y_char = itos[y]
    print("---------------")
    print(f"bigram example {i + 1} is: {x_char} - {y_char} at indexes {x}, {y}")
    print(f"input to the neural net is: {x_char} at index {x}")
    print(f"output probabilities from the neural net are: {probs[i]}")

    prob_output = probs[i, y]
    print(f"probability assigned to the correct character by the neural net is: {prob_output}")
    log_prob = torch.log(prob_output).item()
    print(f"log likelihood is: {log_prob:.4f}")
    nll = -log_prob
    print(f"negative log likelihood is: {nll:.4f}")
    nlls[i] = nll

print("---------------")
print(f"average nll across all bigrams is: {nlls.mean().item():.4f}")




---------------
bigram example 1 is: . - e at indexes 0, 5
input to the neural net is: . at index 0
output probabilities from the neural net are: tensor([0.0103, 0.0031, 0.0883, 0.0488, 0.0064, 0.0219, 0.0265, 0.2235, 0.0110,
        0.0121, 0.0164, 0.0135, 0.0399, 0.0085, 0.0322, 0.0035, 0.0115, 0.0387,
        0.0624, 0.0188, 0.0767, 0.0645, 0.0081, 0.0645, 0.0121, 0.0676, 0.0093],
       grad_fn=<SelectBackward0>)
probability assigned to the correct character by the neural net is: 0.0219158623367548
log likelihood is: -3.8205
negative log likelihood is: 3.8205
---------------
bigram example 2 is: e - m at indexes 5, 13
input to the neural net is: e at index 5
output probabilities from the neural net are: tensor([0.0090, 0.0368, 0.0510, 0.0282, 0.0126, 0.1201, 0.0101, 0.0193, 0.0131,
        0.0238, 0.0227, 0.1325, 0.0155, 0.0319, 0.0023, 0.0075, 0.0407, 0.0083,
        0.0127, 0.0350, 0.0064, 0.0162, 0.0074, 0.0068, 0.0874, 0.1997, 0.0429],
       grad_fn=<SelectBackward0>)
probabil

In [12]:
# Now, we have to make the training loop.

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [13]:
for k in range(1):
    # forward pass
    x_oh = F.one_hot(xs, num_classes=27).float()
    logits = x_oh @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    # loss calculation
    nlls = torch.zeros(xs.shape[0])
    for i in range(xs.shape[0]):
        x = xs[i].item() # input char index
        y = ys[i].item() # label char index
        prob_output = probs[i, y]
        log_prob = torch.log(prob_output)
        nll = -log_prob
        nlls[i] = nll

    loss = nlls.mean()
    print(loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # update weights using gradient descent
    W.data += -10 * W.grad

3.758953094482422


In [14]:
for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

njglu.
oadvqroz.
ixzydtkwzwlgnrpsgqbmfhdknmripcszxjbozw.
nbmhkkhwfjvidkledeshqqyrzglwblbuyytiiiccjwsihqzqrzw.
zieckkvijsz.
